#Overview

This notebook presents a systematic benchmark comparing two state-of-the-art instance segmentation systems — **YOLO11-seg (Ultralytics)** and **SAM3 (Segment Anything Model, Meta)** — across different hardware configurations and model sizes.

Rather than evaluating accuracy in isolation, the goal is to characterize the full latency, throughput, memory, and detection behavior of each system under realistic video inference conditions.

The benchmark was run on a 200-frame video sequence and repeated across CPU and GPU backends.

Three YOLO11-seg model sizes were tested (yolo11n-seg, yolo11m-seg, yolo11x-seg) alongside a single SAM3 configuration running on GPU in automatic segmentation mode.

# Setup & Installation

In [ ]:
# Install dependencies (run once)
!pip install ultralytics opencv-python-headless matplotlib numpy psutil gputil torch torchvision -q

In [ ]:
!pip install git+https://github.com/huggingface/transformers.git
!pip install accelerate requests pillow
!pip install --upgrade transformers --break-system-packages

## Download video

In [ ]:

import gdown

file_id = "13d-WecRvyTozDcQrHxLV1t_e5GQzPiPz"
video_path = '/content/shinjuku.mp4'

url = f'https://drive.google.com/uc?id={file_id}'
gdown.download(url, video_path, quiet=False)


## Preprocessed Example mp4 file

ex_file_id = "1Yy74XqnVOCdRxBxSuw7o0X4iI0BL2Wav"
video_path_preprocessed = '/content/shinjuku_preprocessed_ex.mp4'

url = f'https://drive.google.com/uc?id={ex_file_id}'
gdown.download(url, video_path_preprocessed, quiet=False)

## Imports

In [ ]:
import os
import time
import json
import warnings
import numpy as np
import cv2
import torch
import psutil
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from pathlib import Path
from collections import defaultdict
from ultralytics import YOLO

warnings.filterwarnings('ignore')

# ── Output dirs ──────────────────────────────────────────────
Path('results').mkdir(exist_ok=True)
Path('output_videos').mkdir(exist_ok=True)

# ── Device availability ───────────────────────────────────────
HAS_CUDA = torch.cuda.is_available()
GPU_NAME = torch.cuda.get_device_name(0) if HAS_CUDA else 'N/A'

print(f'PyTorch  : {torch.__version__}')
print(f'CUDA     : {HAS_CUDA}')
print(f'GPU      : {GPU_NAME}')
print(f'CPU cores: {psutil.cpu_count(logical=True)}')

## Configs

In [ ]:
# ─── USER CONFIG ─────────────────────────────────────────────────────────────
VIDEO_PATH     = video_path
MAX_FRAMES     = 200                # set None to process full video
CONF_THRESH    = 0.30
IOU_THRESH     = 0.45
TRACKER        = 'bytetrack.yaml'   # or 'botsort.yaml'
CLASS_FILTER   = None               # e.g. [0] for 'person' only, None = all
SAVE_VIDEO     = True
# ─────────────────────────────────────────────────────────────────────────────

DEVICES = {'CPU': 'cpu'}
if HAS_CUDA:
    DEVICES['GPU'] = 0
print('Devices to benchmark:', list(DEVICES.keys()))

## Load Video

We are going to use the video /content/shinjuku.mp4, which has a resolution of 1280×606. The video contains 631 frames and runs at 30.0 FPS.

**Q: If the video has 631 frames at 30 FPS, what is its total duration?**

In [ ]:
cap = cv2.VideoCapture(VIDEO_PATH)
TOTAL_FRAMES = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
FPS_SRC      = cap.get(cv2.CAP_PROP_FPS)
W            = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
H            = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
cap.release()
print(f'Video: {VIDEO_PATH}  |  {W}x{H}  |  {TOTAL_FRAMES} frames  |  {FPS_SRC:.1f} fps')

## GPU memory helpers

In [ ]:
def get_gpu_memory_mb() -> float:
    """Allocated VRAM in MB (0 if no CUDA)."""
    if HAS_CUDA:
        return torch.cuda.memory_allocated() / 1e6
    return 0.0

def get_cpu_ram_mb() -> float:
    """Current process RSS in MB."""
    return psutil.Process().memory_info().rss / 1e6

def reset_gpu_stats():
    if HAS_CUDA:
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats()

#1 - Tracking algorithms (Bytrack vs BotSort)

## How both work (shared foundations)

Both algorithms follow the same general loop every frame:

1. Run a detector (YOLO, etc.) to get bounding boxes + confidence scores
2. Use a **[Kalman filter](https://en.wikipedia.org/wiki/Kalman_filter)** to predict where existing tracks will be this frame
3. **Associate** detections to tracks using a cost matrix + [Hungarian algorithm](https://en.wikipedia.org/wiki/Hungarian_algorithm).

4. Update matched tracks, initialize new ones, delete lost ones

The key innovations are *how* they handle the association step.



## 1.1 Hungarian algorithm

The **Hungarian algorithm** is a method used in **optimization problems**—specifically, it solves the **assignment problem**.

### What is the assignment problem?

It’s about assigning a set of tasks to a set of agents (like workers, machines, or jobs) in a way that:

* Each agent gets exactly one task
* Each task is assigned to exactly one agent
* The **total cost is minimized** (or total profit is maximized)

---

### Core idea

The Hungarian algorithm finds the **optimal assignment** by working with a **cost matrix** (a table where each entry represents the cost of assigning a specific agent to a specific task).

---

### How it works (simplified steps)

1. **Create the cost matrix**
   Rows = agents, columns = tasks.

2. **Row reduction**
   Subtract the smallest value in each row from all elements of that row.

3. **Column reduction**
   Subtract the smallest value in each column from all elements of that column.

4. **Cover all zeros with minimum number of lines**
   Use horizontal/vertical lines to cover all zero values.

5. **Check optimality**

   * If number of lines = number of rows (or columns), you have an optimal assignment.
   * If not, adjust the matrix and repeat.

6. **Make assignments**
   Choose zeros such that no two are in the same row or column.

---

### Why it’s useful

* Guarantees the **optimal solution**
* Runs in polynomial time (efficient compared to brute force)
* Widely used in:

  * Job scheduling
  * Resource allocation
  * Matching problems (e.g., workers ↔ tasks)

---
Example (simple idea)
---

Each cell shows how many minutes driver takes to reach that car.

**Goal**: find the assignment that minimizes total time.


<u>The problem</u>
---
Assign each driver to exactly one car, minimizing total travel time . **Hungarian is O(n³)**.

<img src="https://raw.githubusercontent.com/EzequielMatiasArevalo/tuia-computer-vision/refs/heads/main/media/pictures/class_7_CPU_vs_GPU_YOLOV11_vs_SAM3/hungarian-0.png">

Step 1 — row reduction
---
Subtract the minimum value in each row from all entries in that row.

**This guarantees at least one zero per row.**

<img src="https://raw.githubusercontent.com/EzequielMatiasArevalo/tuia-computer-vision/refs/heads/main/media/pictures/class_7_CPU_vs_GPU_YOLOV11_vs_SAM3/hungarian-1.png">

The total cost doesn't change — subtracting a constant from a row shifts all assignments equally.

Step 2 — column reduction
---

Subtract the minimum value in each column from all entries in that column.

**Adds more zeros for the algorithm to work with.**

<img src="https://raw.githubusercontent.com/EzequielMatiasArevalo/tuia-computer-vision/refs/heads/main/media/pictures/class_7_CPU_vs_GPU_YOLOV11_vs_SAM3/hungarian-2.png">

Now we have 6 zeros in the matrix.

Step 3 — cover all zeros with minimum lines
---
Find the fewest lines (horizontal or vertical) that cover every zero.

If the number of lines equals n=4, we can read the optimal assignment.

Here we need only 3 lines — **not enough yet**.

<img src="https://raw.githubusercontent.com/EzequielMatiasArevalo/tuia-computer-vision/refs/heads/main/media/pictures/class_7_CPU_vs_GPU_YOLOV11_vs_SAM3/hungarian-4.png">

Since 3 < 4, the current matrix doesn't yet have enough independent zeros for a full assignment.

Min uncovered value = 1

Step 4 — adjust the matrix
---

Subtract the minimum uncovered value (1) from all **uncovered cells**.

Add it to **doubly-covered** cells.

Singly-covered cells stay the same.
This creates new zeros without breaking existing ones.

<img src="https://raw.githubusercontent.com/EzequielMatiasArevalo/tuia-computer-vision/refs/heads/main/media/pictures/class_7_CPU_vs_GPU_YOLOV11_vs_SAM3/hungarian-5.png">

Step 5 — optimal assignment found!
---

We can now cover all zeros with 4 lines (= n).

Find a perfect matching: one zero per row and per column.

Total time = 3 + 8 + 5 + 9 = 25 minutes.

<img src="https://raw.githubusercontent.com/EzequielMatiasArevalo/tuia-computer-vision/refs/heads/main/media/pictures/class_7_CPU_vs_GPU_YOLOV11_vs_SAM3/hungarian-6.png">


Alex → Car A  3 min
Ben → Car C  8 min
Chloe → Car B  5 min
Daisy → Car D  9 min
Total: 25 min ✓

Best possible — any other combo takes longer.


## 1.2 The Kalman filter

**The core idea in one sentence:** a Kalman filter is just a *weighted average* of two noisy estimates — your model's prediction and the sensor's reading — where the weights are determined by how uncertain each source is.

**The predict step** uses your motion model (physics, constant velocity, whatever you know) to project the current belief forward in time. Uncertainty always *grows* here because models are imperfect (process noise Q).

This is why a Kalman filter never "freezes" — it keeps accounting for the fact that the world keeps moving.

**The update step**
---
It is where the magic happens.

When a measurement z arrives, you compute the Kalman gain:

```
K = σ²_pred / (σ²_pred + R)
```

K is a number between 0 and 1.

If your prediction is very uncertain (large σ²) and the sensor is precise (small R), K → 1 and you move almost entirely to the measurement.

If the opposite, K → 0 and you mostly ignore the sensor.
The new mean and variance:

```
μ_post  = μ_pred + K · (z − μ_pred)     ← weighted correction
σ²_post = (1 − K) · σ²_pred             ← always smaller than σ²_pred
```

The posterior variance `(1-K)·σ²_pred` is *always smaller* than either input variance.

This is the algebraic proof that fusing two uncertain sources makes you more certain.

---
**Why it works for tracking (ByteTrack / BoT-SORT connection)**
---

In multi-object tracking, the Kalman filter state is `[x, y, vx, vy]` (position + velocity).

The `predict` step uses the constant-velocity model to guess where the bounding box will be next frame.

The `update` step fuses that with the detector's actual bounding box output.

This is what lets trackers survive frames where the detector misses an object — the Kalman prediction keeps the track alive.

<img src="https://raw.githubusercontent.com/EzequielMatiasArevalo/tuia-computer-vision/refs/heads/main/media/pictures/class_7_CPU_vs_GPU_YOLOV11_vs_SAM3/kalman-filter.png">

---

## 1.3 ByteTrack — every detection counts

ByteTrack's core idea: **don't throw away low-confidence detections**. Standard trackers discard anything below a score threshold, losing useful signal for occluded or partially visible objects.

Instead, ByteTrack runs **two association passes**:

**Pass 1** — match tracks to *high-confidence* detections (score ≥ τ_high, typically 0.6) using IoU-based cost + Hungarian algorithm. Most tracks get resolved here.

**Pass 2** — take unmatched tracks from Pass 1, and try to match them against *low-confidence* detections (τ_low ≤ score < τ_high). This is where occluded objects hiding behind partially visible detections get recovered.

Identity is maintained **purely through motion** (Kalman + IoU). No appearance features, no camera model — which is why it's extremely fast.

---

## 1.4 BoT-SORT — motion + appearance + camera awareness

BoT-SORT was designed to fix two specific weaknesses of ByteTrack:

**1. Camera motion breaks IoU.** If the camera pans, every track's Kalman prediction is wrong relative to the frame — IoU collapses even for correctly tracked objects. BoT-SORT adds **Global Motion Compensation (GMC)** using ECC (Enhanced Correlation Coefficient) to estimate the camera warp each frame and correct the Kalman state before association.

**2. IoU alone can't re-identify after long occlusion.** BoT-SORT adds a **ReID model** (typically a lightweight CNN) that extracts an appearance embedding for each detection. The association cost matrix is then a *weighted fusion* of IoU cost and ReID cosine distance:

```
cost = α · (1 - IoU) + (1 - α) · cosine_distance(ReID_query, ReID_gallery)
```

Each track maintains a gallery of recent embeddings, updated after every confirmed match.







---

## 1.5 Comparison table

| Property | ByteTrack | BoT-SORT |
|---|---|---|
| Low-score recovery | ✅ Two-pass split | Uses score threshold too |
| Camera motion handling | ❌ None | ✅ GMC / ECC warp |
| Appearance features | ❌ IoU only | ✅ ReID embedding |
| Cost function | IoU | α·IoU + (1−α)·ReID |
| Inference speed | Very fast | Slower (ReID + GMC overhead) |
| Best for | Static cameras, crowds | Moving cameras, long occlusions |
| ID switches | More on moving cameras | Fewer, especially post-occlusion |

---

## When to use each

**ByteTrack** is the default choice when you need real-time performance on a static camera — surveillance, sports analytics, retail footfall. It punches far above its weight for the compute cost.

**BoT-SORT** is the right pick when your camera moves (drones, vehicles, handheld), when you need to re-identify objects after long disappearances, or when ID-switch errors carry a real cost (e.g., sports player tracking, autonomous driving).

# 2 - YOLO11 — Instance Segmentation + Multi-Object Tracking

This notebook benchmarks **Ultralytics YOLO11** (segmentation variant) with **ByteTrack** for multi-object tracking on video sequences.

Evaluated metrics:
- **Inference time** per frame (CPU & GPU)
- **FPS** (frames per second)
- **GPU memory usage** (VRAM)
- **mIoU** (vs a reference mask, if available)
- **Track count stability** across frames

> Results are saved to `results/yolov11_metrics.json` for cross-model comparison.

## 2.1 - Core benchmark function

Common YOLOv11 input/inference parameters in [Ultralytics YOLO11 Docs](https://docs.ultralytics.com/models/yolo11):

| Parameter      | Type             | Default        | Description                                                                                         |
| -------------- | ---------------- | -------------- | --------------------------------------------------------------------------------------------------- |
| `source`       | `str`            | —              | Input source: image, video, webcam, folder, URL, RTSP stream, etc.                                  |
| `imgsz`        | `int` or `(h,w)` | `640`          | Input image size. Larger values improve small-object detection but increase latency and VRAM usage. |
| `conf`         | `float`          | `0.25`         | Confidence threshold. Predictions below this score are discarded.                                   |
| `iou`          | `float`          | `0.7`          | IoU threshold for Non-Maximum Suppression (NMS). Controls duplicate box suppression.                |
| `device`       | `str/int`        | `None`         | Execution device: `cpu`, `0`, `0,1`, `cuda:0`, etc.                                                 |
| `half`         | `bool`           | `False`        | Enables FP16 inference for faster GPU execution.                                                    |
| `batch`        | `int`            | `1`            | Batch size for inference on videos/directories.                                                     |
| `max_det`      | `int`            | `300`          | Maximum detections per image.                                                                       |
| `classes`      | `list[int]`      | `None`         | Filter detections by class IDs.                                                                     |
| `agnostic_nms` | `bool`           | `False`        | Apply class-agnostic NMS.                                                                           |
| `augment`      | `bool`           | `False`        | Test-time augmentation during inference.                                                            |
| `visualize`    | `bool`           | `False`        | Visualize feature maps.                                                                             |
| `show`         | `bool`           | `False`        | Display predictions in a window.                                                                    |
| `save`         | `bool`           | `False`        | Save output predictions.                                                                            |
| `save_txt`     | `bool`           | `False`        | Save detections in YOLO text format.                                                                |
| `save_conf`    | `bool`           | `False`        | Save confidence scores with labels.                                                                 |
| `stream`       | `bool`           | `False`        | Stream inference results instead of returning all at once.                                          |
| `vid_stride`   | `int`            | `1`            | Process every N-th frame in video.                                                                  |
| `tracker`      | `str`            | `botsort.yaml` | Tracker config for multi-object tracking.                                                           |

Important parameter relationships:

* Lower `conf` → more detections, more false positives.
* Lower `iou` → more aggressive duplicate removal.
* Higher `imgsz` → better accuracy on small objects but slower inference.
* `half=True` only helps on supported GPUs (CUDA FP16). ([Ultralytics Docs][1])

[1]: https://docs.ultralytics.com/es/usage/cfg?utm_source=chatgpt.com "Configuración | Documentación de Ultralytics"


In [ ]:
MODEL_NAMES     = ['yolo11n-seg.pt','yolo11m-seg.pt', 'yolo11x-seg.pt']   # options: yolo11n/s/m/l/x-seg.pt


In [ ]:
def run_yolo11_benchmark(model, device_label: str, device) -> dict:
    """
    Run YOLO11-seg + ByteTrack on the video and collect per-frame metrics.
    Returns a dict with lists: inference_ms, n_masks, n_tracks, gpu_mem_mb.
    """
    print(f'\n{'='*60}')
    print(f'  YOLO11  |  device={device_label}  |  model={model}')
    print(f'{'='*60}')

    reset_gpu_stats()
    model = YOLO(model)

    metrics = defaultdict(list)
    cap = cv2.VideoCapture(VIDEO_PATH)
    n_frames = MAX_FRAMES or TOTAL_FRAMES

    # optional video writer
    vout = None
    if SAVE_VIDEO:
        out_path = f'output_videos/{m}_{device_label.lower()}.mp4'
        vout = cv2.VideoWriter(out_path,
                               cv2.VideoWriter_fourcc(*'mp4v'),
                               FPS_SRC, (W, H))

    # warm-up (GPU JIT / kernel compilation)
    ret, warm_frame = cap.read()
    if ret:
        _ = model(warm_frame, device=device, verbose=False)
        cap.set(cv2.CAP_PROP_POS_FRAMES, 0)

    for frame_idx in range(n_frames):
        ret, frame = cap.read()
        if not ret:
            break

        t0 = time.perf_counter()
        results = model.track(
            frame,
            persist=True,
            tracker=TRACKER,
            device=device,
            conf=CONF_THRESH,
            iou=IOU_THRESH,
            classes=CLASS_FILTER,
            verbose=False
        )
        t1 = time.perf_counter()

        inf_ms = (t1 - t0) * 1000
        r = results[0]

        n_masks  = len(r.masks) if r.masks is not None else 0
        track_ids = (r.boxes.id.cpu().numpy().tolist()
                     if r.boxes.id is not None else [])
        n_tracks = len(set(track_ids))

        metrics['inference_ms'].append(inf_ms)
        metrics['n_masks'].append(n_masks)
        metrics['n_tracks'].append(n_tracks)
        metrics['gpu_mem_mb'].append(get_gpu_memory_mb())
        metrics['cpu_ram_mb'].append(get_cpu_ram_mb())

        if SAVE_VIDEO and vout:
            annotated = r.plot()
            cv2.putText(annotated,
                        f'{device_label}  {inf_ms:.1f}ms  masks:{n_masks}  tracks:{n_tracks}',
                        (10, 28), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0,255,128), 2)
            vout.write(annotated)

        if frame_idx % 30 == 0:
            print(f'  frame {frame_idx:>4d}/{n_frames}  '
                  f'inf={inf_ms:6.1f}ms  masks={n_masks}  tracks={n_tracks}'
                  f'  vram={get_gpu_memory_mb():.0f}MB')

    cap.release()
    if vout:
        vout.release()

    metrics = dict(metrics)
    arr = np.array(metrics['inference_ms'])
    metrics['summary'] = {
        'mean_ms'   : float(np.mean(arr)),
        'median_ms' : float(np.median(arr)),
        'std_ms'    : float(np.std(arr)),
        'p95_ms'    : float(np.percentile(arr, 95)),
        'min_ms'    : float(np.min(arr)),
        'max_ms'    : float(np.max(arr)),
        'fps'       : float(1000.0 / np.mean(arr)),
        'n_frames'  : len(arr),
        'peak_gpu_mb': float(np.max(metrics['gpu_mem_mb'])),
        'peak_cpu_mb': float(np.max(metrics['cpu_ram_mb'])),
    }

    s = metrics['summary']
    print(f'\n  ✔ Done  mean={s["mean_ms"]:.1f}ms  '
          f'FPS={s["fps"]:.1f}  peak_vram={s["peak_gpu_mb"]:.0f}MB')
    return metrics

## 2.2 - Run benchmarks

YOLO11 uses the `imgsz` parameter to define the model input resolution.

Typical values:

| `imgsz`        | Use case                          |
| -------------- | --------------------------------- |
| `320`          | Very fast inference, low accuracy |
| `640`          | Default / balanced                |
| `800` / `1024` | Better small-object detection     |
| `1280+`        | High-detail or aerial imagery     |


YOLO11 internally resizes images before inference. For detection models, Ultralytics typically uses **letterboxing** (resize + padding) to preserve aspect ratio.

Important notes:

* Default input size for pretrained YOLO11 models is usually:
  640x640


* Larger `imgsz`:

  * improves small-object detection
  * increases VRAM usage
  * increases latency roughly quadratically

* `imgsz` should usually be a multiple of:
  32
  because YOLO downsamples feature maps by stride 32.

* Non-square inputs are supported:

```python
imgsz=(1280, 736)
```

* During preprocessing:

  * original image is resized
  * padding may be added
  * bounding boxes are automatically remapped to original coordinates

Example for small objects:

```python
results = model.predict(
    source="4k_frame.jpg",
    imgsz=1280
)
```

Example for real-time webcam:

```python
results = model.predict(
    source=0,
    imgsz=416
)
```


In [ ]:
all_results = {}
for m in MODEL_NAMES:

  # Persist results for the comparison notebook
  all_results[m]={}
  for label, device in DEVICES.items():
      all_results[m][label] = run_yolo11_benchmark(m,label, device)

  # Persist results for the comparison notebook
  save_data = {k: v['summary'] for k, v in all_results[m].items()}
  with open(f'results/{m}_metrics.json', 'w') as f:
      json.dump({'model': 'YOLO11-seg', 'config': m,
                'results': save_data}, f, indent=2)

  print(f'\nSaved → results/{m}_metrics.json')

In [ ]:
import cv2
import matplotlib.pyplot as plt

fig , axes = plt.subplots(1,3,figsize=(20,20))

for i,m in enumerate(MODEL_NAMES):
  video_path = f"output_videos/{m}_gpu.mp4"
  cap = cv2.VideoCapture(video_path)
  ret, frame = cap.read()

  if ret:
      frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
      axes[i].imshow(frame)
      axes[i].set_title(m)
      axes[i].axis('off')

  cap.release()


## 2.3 - Per-device plots

### 2.3.1 Inference Time Benchmark

These two graphs compare **per-frame inference latency** across three YOLO11 segmentation model sizes (n, m, x) under two hardware conditions.

In [ ]:

import random

colors = np.random.rand(len(MODEL_NAMES),3 )
PALETTE = {'CPU': '#4C9BE8', 'GPU': '#F2A541'}
# generate random color by model
PALETTE_BY_MODEL = {
   m : "#{:06x}".format(random.randint(0, 0xFFFFFF))  for m in MODEL_NAMES
}

def smooth(x, w=7):
    return np.convolve(x, np.ones(w)/w, mode='valid')

n_devices = len(DEVICES)
print(n_devices)
fig, axes = plt.subplots(n_devices, 1,
                          figsize=(14, 4.5 * n_devices), sharex=False)
if n_devices == 1:
    axes = [axes]

for m in MODEL_NAMES:
  # ── 1. Inference time over frames ──────────────────────────────

  metrics = all_results[m]

  model_name_base = m.split(".")[0].split("-")[0]
  for ax, (label, data) in zip(axes, metrics.items()):
      ms   = np.array(data['inference_ms'])
      xs   = np.arange(len(ms))
      col  = PALETTE_BY_MODEL.get(m, 'steelblue')
      ax.fill_between(xs, ms, alpha=0.15, color=col)
      ax.plot(xs, ms, color=col, alpha=0.4, lw=0.8, label=f'{model_name_base} raw')
      ax.plot(np.arange(len(smooth(ms))), smooth(ms),
              color=col, lw=2)
      ax.axhline(data['summary']['mean_ms'], ls='--', color=col,
                lw=1.4, label=f'{model_name_base} mean = {data["summary"]["mean_ms"]:.1f} ms')
      ax.set_title(f'Yolo11-seg  |  {label}  —  Inference Time per Frame', fontsize=13, fontweight='bold')
      ax.set_ylabel('Latency (ms)')
      ax.set_xlabel('Frame index')
      ax.legend()
      ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(f'results/yolo11_inference_time.png', dpi=150, bbox_inches='tight')
plt.show()

#### Analysis

---

### Top graph — CPU

| Model | Mean latency |
|---|---|
| yolo11n | 135.0 ms |
| yolo11m | 730.2 ms |
| yolo11x | 1842.5 ms |

- **yolo11n** is fast and stable on CPU (~135 ms ≈ ~7 FPS)
- **yolo11m** is ~5× slower than n
- **yolo11x** is extremely slow (~1.8 s/frame), making it **unusable for real-time CPU inference**
- The large **shaded bands** around yolo11x indicate high variance — latency spikes heavily (up to ~4500 ms) around frames 130–160, likely due to complex scenes with many objects/masks to process
- yolo11n has almost no visible band → very consistent timing

---

### Bottom graph — GPU

| Model | Mean latency |
|---|---|
| yolo11n | 17.9 ms (~56 FPS) |
| yolo11m | 23.4 ms (~43 FPS) |
| yolo11x | 45.7 ms (~22 FPS) |

- **All three models are viable for real-time** on GPU
- The gap between n and x is only ~2.5× (vs ~13× on CPU) — GPU parallelism dramatically narrows the difference
- There's a **large spike around frame ~110–115** across all models simultaneously → this is almost certainly a scene with unusually high object density or large masks, not a model artifact
- After frame ~130, yolo11x stabilizes back to its mean

---

### Key takeaways

- **GPU is non-negotiable for yolo11m/x** in real-time applications
- **yolo11n on GPU** is the sweet spot for edge deployment (~18 ms, stable)
- The correlated spike at frame ~110 across all models tells you the bottleneck there is **the input data** (hard frame), not the model itself
- CPU variance in yolo11x suggests it's sensitive to scene complexity — likely driven by the NMS + mask postprocessing cost scaling with detection count

### 2.3.2 Detected Masks & Active Tracks per Frame


This is a 2×2 grid comparing **what each model detects**, not how fast it runs. Hardware (CPU/GPU) is in the rows, metric type in the columns.



In [ ]:
# ── 2. Masks & tracks per frame ────────────────────────────────
fig, axes = plt.subplots(2, 2,
                          figsize=(16, 4 * len(all_results)))
if len(all_results) == 1:
    axes = [axes]

for m in MODEL_NAMES:
  # ── 1. Inference time over frames ──────────────────────────────

  metrics = all_results[m]

  model_name_base = m.split(".")[0].split("-")[0]

  for row, (label, data) in enumerate(metrics.items()):
      col = PALETTE_BY_MODEL.get(m, 'steelblue')
      xs  = np.arange(len(data['n_masks']))

      for ax, key, title, color in zip(
          axes[row],
          ['n_masks', 'n_tracks'],
          ['Detected Masks / Frame', 'Active Tracks / Frame'],
          [col, col]
      ):
          vals = np.array(data[key])
          ax.step(xs, vals, where='mid', color=color, lw=1.5)
          ax.fill_between(xs, vals, step='mid', alpha=0.15, color=color)
          ax.axhline(np.mean(vals), ls='--', color=col, lw=1.2,
                    label=f'{m} mean = {np.mean(vals):.1f}')
          ax.set_title(f'YOLO11 | {label} — {title}', fontweight='bold')
          ax.set_xlabel('Frame')
          ax.legend()
          ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('results/yolo11_masks_tracks.png', dpi=150, bbox_inches='tight')
plt.show()

#### Analysis
---

### What each plot shows

| Column | Metric |
|---|---|
| Left | **Detected Masks/Frame** — raw segmentation output count per frame |
| Right | **Active Tracks/Frame** — objects being tracked (post-tracker filtering) |

---

### The most important observation

**The CPU and GPU plots are nearly identical** — the curves are virtually the same across all 4 panels. This confirms that hardware doesn't affect detection results, only speed. The model weights determine what gets detected.

---

### Per-model behavior

**yolo11n (blue) — mean 3.7**
- Detects very few objects per frame, consistently low
- High confidence threshold effectively — it only fires on obvious, high-confidence detections
- Very stable, little variance

**yolo11x (purple) — mean 13.9**
- Detects significantly more objects than `n`

**yolo11m (green) — mean 16.0**
- Detects the *most* on average, even more than `x`
- This is counterintuitive at first glance

---

### Why does yolo11m detect more than yolo11x?

This is the most interesting finding in these graphs. Possible explanations:

- **yolo11x is more selective** — it has higher effective precision, so it suppresses borderline detections that `m` passes through
- **Overfitting or calibration difference** — larger models can be more conservative if their confidence scores are better calibrated

This does **not** mean `m` is better than `x` — it may mean `x` is more precise (fewer false positives), while `m` casts a wider net.

---

### Detected Masks vs. Active Tracks (left vs. right)

The two columns are almost identical in shape, which means:
- The tracker is keeping up with detections — very little track loss
- Minimal ID-switch noise visible
- The tracker isn't adding or suppressing much relative to raw detections

If there were significant differences between the two columns, it would indicate tracker instability (lost tracks, ghost tracks, etc.).

---

### Scene structure visible in the signal

The detection counts follow a clear pattern across all models:
- **Frames 0–50**: lower count, simpler scene
- **Frames 75–100**: scene gets busier, spike in detections
- **Frames 150–200**: peak complexity, highest counts (up to 25+ objects)

This matches the latency spike you saw at ~frame 110 in the previous graphs — dense scenes drive both higher latency and higher detection counts simultaneously.

### 2.3.3 CPU vs GPU summary bar chart
Three metrics side by side, all three models, both hardware backends.


In [ ]:
# ── 3. CPU vs GPU summary bar chart (if both available) ────────
print(all_results.keys())
if len(all_results[MODEL_NAMES[0]]) > 1:
    metrics_keys = ['mean_ms', 'p95_ms', 'fps']
    titles   = ['Mean latency (ms)', 'P95 latency (ms)', 'FPS (1000/mean_ms)']
    colors   = [PALETTE_BY_MODEL.get(m, 'steelblue') for m in MODEL_NAMES]

    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    for ax, key, title in zip(axes, metrics_keys, titles):
        for m in MODEL_NAMES:

          model_base_name = m.split(".")[0].split("-")[0]

          tmp_vals = all_results[m]
          labels  = list(tmp_vals.keys())

          vals = [tmp_vals[l]['summary'][key] for l in labels]
          bars = ax.bar(labels, vals, color=PALETTE_BY_MODEL.get(m, 'steelblue'), width=0.5, edgecolor='white', linewidth=1.5)
          ax.bar_label(bars, fmt=f'{model_base_name} | %.1f ', padding=0, fontweight='bold')
          ax.set_title(title, fontsize=12, fontweight='bold')
          #ax.set_ylim(0, max(vals) * 1.4)
          ax.grid(axis='y', alpha=0.3)
          ax.spines[['top','right']].set_visible(False)

    fig.suptitle('YOLO11-seg  |  CPU vs GPU Summary', fontsize=14, fontweight='bold', y=1.02)
    plt.tight_layout()
    plt.savefig('results/yolo11_cpu_vs_gpu.png', dpi=150, bbox_inches='tight')
    plt.show()
else:
    print('Only one device available — skipping CPU vs GPU comparison plot.')

In [ ]:
def remove_outliers_percentile(arr, low=5, high=95):
    lo, hi = np.percentile(arr, [low, high])
    return arr[(arr >= lo) & (arr <= hi)]

def box_plot(data: dict, remove_outliers: bool = True, model_name : str=None):
  # ── 4. Latency distribution (box plot + strip) ─────────────────
  fig, ax = plt.subplots(figsize=(8, 5))

  if remove_outliers:
    data_list  = [remove_outliers_percentile(np.array(data[l]['inference_ms'])) for l in data]
  else:
    data_list  = [np.array(data[l]['inference_ms']) for l in data]

  colors_box = [PALETTE.get(l, 'steelblue') for l in data]

  bp = ax.boxplot(data_list, patch_artist=True, notch=True,
                  medianprops=dict(color='white', linewidth=2))
  for patch, col in zip(bp['boxes'], colors_box):
      patch.set_facecolor(col)
      patch.set_alpha(0.7)

  # jitter overlay
  for i, (data_arr, col) in enumerate(zip(data_list, colors_box), start=1):
      jitter = np.random.default_rng(0).normal(0, 0.07, len(data_arr))
      ax.scatter(i + jitter, data_arr, alpha=0.25, s=12, color=col)

  ax.set_xticklabels(list(data.keys()), fontsize=12)
  ax.set_ylabel('Latency (ms)', fontsize=11)
  ax.set_title(f'{model_name}  |  Latency Distribution (all frames)', fontsize=13, fontweight='bold')
  ax.grid(axis='y', alpha=0.3)
  plt.tight_layout()
  plt.savefig('results/yolo11_latency_dist.png', dpi=150, bbox_inches='tight')
  plt.show()

In [ ]:
for m in MODEL_NAMES:
  box_plot(all_results[m], remove_outliers=True, model_name=m)


#### Analysis
---

### Mean Latency (ms)

| Model | CPU | GPU | Speedup |
|---|---|---|---|
| yolo11n | 135.0 ms | ~17.9 ms | ~7.5× |
| yolo11m | 730.2 ms | ~23.4 ms | ~31× |
| yolo11x | 1842.5 ms | ~45.3 ms | ~41× |

The GPU bars are so small they're nearly invisible compared to CPU — the scale is dominated by yolo11x on CPU (1842 ms).

---

### P95 Latency (ms)

| Model | CPU | GPU |
|---|---|---|
| yolo11n | 260.7 ms | ~20.5 ms |
| yolo11m | 984.1 ms | ~20.5 ms |
| yolo11x | 2731.1 ms | ~20.5 ms |

**Key insight:** P95 is significantly higher than mean for all CPU models, especially yolo11x (2731 vs 1842 ms — the tail is ~48% worse than mean). This confirms what the per-frame graphs showed: **CPU latency has long tails** driven by complex scenes. On GPU, P95 ≈ mean, meaning latency is highly predictable regardless of scene complexity.

---

### FPS (1000 / mean_ms)

| Model | CPU | GPU |
|---|---|---|
| yolo11n | 7.4 FPS | ~55.9 FPS |
| yolo11m | 1.4 FPS | ~42.7 FPS |
| yolo11x | 0.5 FPS | **21.9 FPS** |

The FPS chart is the most visually striking — on CPU, only yolo11n breaks even near real-time tolerance, and yolo11x is essentially a slideshow at 0.5 FPS. On GPU, **even the heaviest model (x) runs at ~22 FPS**, and n approaches 56 FPS.

---

### Bottom line

| Decision | Recommendation |
|---|---|
| Real-time, edge CPU | yolo11n only (7 FPS, barely usable) |
| Real-time, GPU | Any model — n for max FPS, x if you need accuracy |
| Latency-critical prod | GPU + yolo11n (predictable P95, ~20 ms) |
| Best accuracy/speed tradeoff GPU | yolo11m (43 FPS, much better detection than n) |

The GPU speedup scales **superlinearly with model size** (7× for n, 41× for x) — larger models benefit disproportionately from GPU parallelism, which is why yolo11x on GPU is practical while yolo11x on CPU is not.

### 2.3.4 Summary

**Latency tails (mean vs P95 on CPU)**

| Model | Mean | P95 | Tail ratio |
|---|---|---|---|
| yolo11n | 135 ms | 261 ms | 1.93× |
| yolo11m | 730 ms | 984 ms | 1.35× |
| yolo11x | 1842 ms | 2731 ms | 1.48× |

yolo11n has the worst tail ratio — its mean is pulled down by fast frames, but complex scenes hit it hard proportionally. yolo11x is already slow everywhere so the tail ratio looks smaller.

**GPU memory is the real constraint for x**
---

yolo11x on GPU needs 385 MB VRAM peak — fine for any modern discrete GPU, but tight on integrated graphics or older mobile GPUs. yolo11n at 50 MB VRAM is virtually free.

**Std dev tells a different story than mean**
---

On CPU, yolo11n std = 98.9 ms against a mean of 135 ms — that's a coefficient of variation of ~73%. The model is fast on average but very unpredictable frame-to-frame. yolo11m by comparison has a CV of ~17%, much more consistent. If you need predictable latency (e.g. pipeline buffering), yolo11m on GPU is the safest choice.

In [ ]:
import pandas as pd

for m in MODEL_NAMES:
  rows = []
  for label, data in all_results[m].items():
      s = data['summary']
      rows.append({
          'Device'     : label,
          'Mean ms'    : round(s['mean_ms'], 2),
          'Median ms'  : round(s['median_ms'], 2),
          'Std ms'     : round(s['std_ms'], 2),
          'P95 ms'     : round(s['p95_ms'], 2),
          'Min ms'     : round(s['min_ms'], 2),
          'Max ms'     : round(s['max_ms'], 2),
          'FPS'        : round(s['fps'], 1),
          'Peak VRAM MB': round(s['peak_gpu_mb'], 1),
          'Peak RAM MB' : round(s['peak_cpu_mb'], 1),
      })

  print("="*50)
  print(m)

  df = pd.DataFrame(rows).set_index('Device')
  df.style.background_gradient(cmap='YlOrRd', axis=0)
  print(df)


# 3 - SAM 3

The default input image size for Segment Anything Model 3 (SAM 3) is **1008x1008** pixels.

However, specific implementations and variants may use different resolutions

Common Input Resolutions:
---

- **Standard Implementation**: The model is optimized for 1008x1008 px resolution.

- **Ultralytics & Transformers Integration**: Often defaults to 1008x1008 pixels.

- **Pretraining Resolution**: The model often uses a pretraining resolution of 336x336 pixels for position embedding initialization.

Key Requirements & ConstraintsSquare Aspect Ratio:
---

- **Images typically must be square (e.g., 1008*1008)**.
- **Divisibility**: For best results, the size should be divisible by the patch size, which is 14 (e.g., 1008/14 = 72).
- **Custom Resolutions**: While custom resolutions (like 512 or 1024 for tiled segmentation) can be used, they may lead to a degradation in accuracy compared to the intended resolution.
- **Automatic Resizing**: Tools like the Sam3Processor on Hugging Face handle resizing original images to the model's required input size and then post-process masks back to the original dimensions.

In [ ]:

from huggingface_hub import hf_hub_download
from google.colab import userdata

userdata.get('HF_TOKEN')
REPO_ID = "facebook/sam3"
FILENAME = "sam3.pt"

hf_hub_download(repo_id=REPO_ID, filename=FILENAME, local_dir="/content/")

In [ ]:
from ultralytics.models.sam import SAM3VideoSemanticPredictor
from ultralytics import SAM

overrides = dict(
    conf=0.5,
    task="segment",
    mode="predict",
    model="sam3.pt",
    half=True,
    save=True,
)

try:
  del predictor
except:
  pass

predictor = SAM3VideoSemanticPredictor(overrides=overrides)



## 3.1 Core benchmark functions

### 3.1.1 Defining hooks to get metrics per frame

In [ ]:
import time
import psutil
import threading
import numpy as np
import torch
import cv2
from typing import Literal
from collections import defaultdict
from ultralytics import YOLO


class ResourceMonitor:
    """Samples CPU % and VRAM MB in a background thread at fixed intervals."""

    def __init__(self, device: Literal["cpu", "gpu"]):
      self.device = device
      self.pre_process_speed = []
      self.inference_speed = []
      self.post_process_speed = []
      self.cpu_pct = []
      self.classes_by_name = []
      self.n_instances_by_class = []
      self.vram_mb = []
      self.n_mask = []
      self.n_tracks = []
      self.peak_gpu_mb = []
      self.peak_cpu_mb = []

    def summary(self):

      return {
          'instances_last_frame': self.n_instances_by_class[-1],
          'classes_last_frame': self.classes_by_name[-1],
          'mean_ms': float(np.mean(self.inference_speed)),
          'median_ms': float(np.median(self.inference_speed)),
          'std_ms': float(np.std(self.inference_speed)),
          'p95_ms': float(np.percentile(self.inference_speed, 95)),
          'min_ms': float(np.min(self.inference_speed)),
          'max_ms': float(np.max(self.inference_speed)),
          'fps': float(1000.0 / np.mean(self.inference_speed)),
          'n_frames': len(self.inference_speed),
          'peak_gpu_mb': float(np.max(self.vram_mb)),
          'peak_cpu_mb': float(np.max(self.cpu_pct)),
      }


def make_yolo_hooks(monitor: ResourceMonitor):
    """
    Returns (pre_hook, post_hook) callables to register on a YOLO model.
    Captures resource usage bracketing each forward pass.
    """

    def post_hook(predictor):
      results = predictor.results[0]

      n_instances_by_class = {}
      for box, clsid in zip(results.boxes.xyxy, results.boxes.cls):
        clsid = int(clsid)
        if clsid not in n_instances_by_class:
          n_instances_by_class[clsid] = 0
        n_instances_by_class[clsid] += 1

      monitor.inference_speed.append(results.speed["inference"])
      monitor.pre_process_speed.append(results.speed["preprocess"])
      monitor.post_process_speed.append(results.speed["postprocess"])
      monitor.n_mask.append(len(results.masks))
      monitor.classes_by_name.append(results.names)
      monitor.n_instances_by_class.append(n_instances_by_class)
      monitor.n_tracks.append(len(results.boxes))
      monitor.cpu_pct.append(get_cpu_ram_mb())
      monitor.vram_mb.append(get_gpu_memory_mb())
      monitor.peak_gpu_mb.append(np.max(monitor.vram_mb))
      monitor.peak_cpu_mb.append(np.max(monitor.cpu_pct))
    return post_hook

In [ ]:
monitor = ResourceMonitor('gpu')
post_hook = make_yolo_hooks(monitor)
predictor.add_callback("on_predict_batch_end", post_hook)
results = predictor(
    source=VIDEO_PATH,
    text=["person", "car"],
    stream=True
)



### 3.1.2 Inference

In [ ]:

MAX_FRAMES=200
for i,r in enumerate(results):
    if i > MAX_FRAMES:
      break

## 3.2 Inference time analysis

**Mean = 1490.7 ms on GPU.**

Compare that to YOLO11x on GPU at 45.7 ms. **SAM3 is ~33× slower** than `YOLO11x` on the same hardware, and **~11x slower** than YOLO11n on CPU (135 ms).

This is a fundamentally different class of model.

In [ ]:
def smooth(x, w=7):
    return np.convolve(x, np.ones(w)/w, mode='valid')

fig, ax = plt.subplots(1, 1,
                          figsize=(14, 4.5 ), sharex=False)

data={
    'inference_ms': monitor.inference_speed,
    'pre_process_ms': monitor.pre_process_speed,
    'post_process_ms': monitor.post_process_speed,
    'total_ms': np.sum([monitor.inference_speed, monitor.pre_process_speed,  monitor.post_process_speed]),
    'mean_total': np.mean([monitor.inference_speed, monitor.pre_process_speed,  monitor.post_process_speed]),
    'cpu_pct': monitor.cpu_pct,
    'vram_mb': monitor.vram_mb,
    'n_masks': monitor.n_mask,
    'n_tracks': monitor.n_tracks,
    'classes_by_name': monitor.classes_by_name,
    'n_instances_by_class': monitor.n_instances_by_class,
    'summary': monitor.summary()
}

label = "gpu"
model_name_base = "YOLO SAM-3"

ms   = np.array(data['inference_ms'])
xs   = np.arange(len(ms))
col  = 'steelblue'
ax.fill_between(xs, ms, alpha=0.15, color=col)
ax.plot(xs, ms, color=col, alpha=0.4, lw=0.8, label=f'{model_name_base} raw')
ax.plot(np.arange(len(smooth(ms))), smooth(ms),
        color=col, lw=2)
ax.axhline(data['summary']['mean_ms'], ls='--', color=col,
          lw=1.4, label=f'{model_name_base} mean = {data["summary"]["mean_ms"]:.1f} ms')
ax.set_title(f'SAM3 |  {label}  —  Inference Time per Frame', fontsize=13, fontweight='bold')
ax.set_ylabel('Latency (ms)')
ax.set_xlabel('Frame index')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(f'results/sam3_inference_time.png', dpi=150, bbox_inches='tight')
plt.show()

#### 3.2.1 GPU — Inference Time per Frame

Latency profile analysis
---

**Range:** ~1000–2600 ms, with most frames between 1100–1700 ms.

The curve has a clear **wave structure** — not random noise, but correlated with scene content:

| Frame range | Latency | Interpretation |
|---|---|---|
| 0–5 | ~1000 ms dip | Warm-up / simple first frame |
| 5–25 | ~2100 ms peak | Dense/complex scene |
| 25–75 | Declining 2100→1100 ms | Scene simplifies |
| 75–125 | Trough ~1050–1150 ms | Simplest frames in video |
| 125–135 | Spike ~1600 ms | Local complexity burst |
| 135–185 | ~1200–1600 ms | Mid-complexity |
| 185–195 | Peak ~2100–2600 ms | Hardest frames in video |


Why SAM3 is this slow
---

SAM is architecturally very different from YOLO:

- **Prompt encoder + image encoder + mask decoder** all run per frame
- The image encoder (ViT-based) dominates cost — it processes the full image at high resolution
- SAM generates **dense, high-quality masks** rather than lightweight bounding-box-anchored masks
- Latency scales with the number of prompted points/boxes if using prompt mode; in automatic mode (likely here), it generates masks for the entire image exhaustively

---

### The shaded band is narrow

Unlike YOLO11x on CPU (huge variance), the raw signal stays close to the smoothed line — the shaded region is thin. This means **SAM3 latency is consistent** — it's always slow, but predictably so. The variance is driven by scene content, not by model instability.

---

### Practical bottom line

| Use case | Verdict |
|---|---|
| Real-time video (≥15 FPS) | Not viable on any hardware with SAM3 |
| Offline batch segmentation | Fine — 1–2 s/frame is acceptable |
| High-quality single-image masking | Ideal use case |
| Combined with YOLO (detect → SAM refine) | Possible offline, not real-time |

---

**YOLO trades mask quality for speed (prototype-based masks), while SAM trades speed for mask fidelity (dense ViT-based segmentation). They solve different problems**


In [ ]:
import seaborn as sns
import pandas as pd

fig, axes = plt.subplots(1, 2,
                          figsize=(16, 4 ))



xs  = np.arange(len(data['n_masks']))

PALETTE_BY_CLASS = {
  "person" : "red",
  "car": "blue"
}

# Loop for n_masks and n_tracks plots
for ax_idx, (ax, key, title, color) in enumerate(zip(
    axes,
    ['n_masks', 'n_tracks'],
    ['Detected Masks / Frame', 'Active Tracks / Frame'],
    [col, col]
)):
    # Plotting of n_masks/n_tracks over frames
    vals = np.array(data[key])
    ax.step(xs, vals, where='mid', color=color, lw=1.5)
    ax.fill_between(xs, vals, step='mid', alpha=0.15, color=color)
    ax.axhline(np.mean(vals), ls='--', color=col, lw=1.2,
              label=f'SAM3 mean = {np.mean(vals):.1f}')

    # Set title, labels, legend, grid for the current subplot
    ax.set_title(f'SAM3 | {label} — {title}', fontweight='bold')
    ax.set_xlabel('Frame')
    ax.legend()
    ax.grid(True, alpha=0.3)


plt.tight_layout()
plt.savefig('results/sam3_masks_tracks.png', dpi=150, bbox_inches='tight')
plt.show()


# Prepare data for plotting the stacked histogram
plot_data = []
for frame_idx in range(len(data['n_instances_by_class'])):
    frame_instance_counts = data['n_instances_by_class'][frame_idx]
    frame_class_map = data['classes_by_name'][frame_idx]

    for class_id_key, count in frame_instance_counts.items():
        class_id = int(class_id_key)
        # Ensure class_id is within bounds of frame_class_map
        class_name = frame_class_map[class_id] if 0 <= class_id < len(frame_class_map) else f"Unknown_class_{class_id}"
        plot_data.append({'frame_idx': frame_idx, 'class_name': class_name, 'count': count})

df_class_counts = pd.DataFrame(plot_data)

# Create the stacked histogram
plt.figure(figsize=(15, 7))
sns.histplot(data=df_class_counts, x='frame_idx', hue='class_name', weights='count', multiple='stack', binwidth=1)
plt.title('Instance Counts per Frame by Class (SAM3)', fontsize=16, fontweight='bold')
plt.xlabel('Frame Index', fontsize=12)
plt.ylabel('Total Instances', fontsize=12)
plt.legend(title='Class', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()

### 3.2.2 Instance Counts per Frame by Class

What the chart shows
---

A stacked bar chart where each bar = one frame, and the two colors represent **two detected classes**:

- **Orange (bottom)** — one class(car), consistently detected
- **Blue (top)** — a second class (person), more variable


<u>Orange layer (Cars) — highly stable</u>
---

Orange(cars) sits between ~15–22 instances across almost all 200 frames with very little variance. This is a class that is **always present in the scene**

<u>Blue layer (persons) — scene-driven variance</u>
---

Blue drives all the variability in total count. It ranges from ~6 to ~40+ instances and tracks the scene complexity curve you've seen in the other graphs:

| Frame range | Blue instances | Pattern |
|---|---|---|
| 0–25 | 30–45 | Dense scene, peak early |
| 25–75 | 10–25 | Scene simplifies |
| 75–110 | 5–15 | Minimum complexity |
| 110–135 | 15–20 | Mid recovery |
| 135–200 | 15–25 | Gradual increase |


<u>The spike at frame ~200</u>
---

The last frame hits **~86 total instances** — roughly double the typical maximum. This is a clear outlier. Possible causes:

- A sudden scene change with many new objects
- SAM3 over-segmenting a complex or cluttered frame (many small regions detected)
- An artifact of the video ending (partial frame, encoding artifact triggering false detections)

Notably, **both layers spike** at frame 200 — orange jumps from ~20 to ~42, blue from ~20 to ~44.

**This suggests it's a genuine scene-level event, not a class-specific glitch**.

<u>Cross-referencing with latency</u>
---

The SAM3 latency graph showed a **peak around frames 185–195** (~2100 ms). The instance count spike at frame 200 aligns — more instances means more mask generation, which directly drives up inference time. This confirms the scene complexity → detection count → latency chain you can observe consistently across all models in this benchmark.


<u>Key SAM3 vs YOLO11 comparison</u>
---

SAM3 detects **35–86 instances per frame** on average. YOLO11m detected ~16 mean, yolo11x ~14.

SAM3 is far more generous in what it segments — this is expected since SAM runs in **automatic everything mode** (no class filter, no confidence threshold in the traditional sense), segmenting every coherent region it finds.

# 4 - YoloV11 (CPU & GPU) vs SAM3 Summary


In [ ]:
if "sam3" in all_results:
  del all_results["sam3"]

all_results["sam3"] = {
    "gpu": data
}

if len(all_results[MODEL_NAMES[0]]) > 1:
    metrics_keys = ['mean_ms', 'p95_ms', 'fps']
    titles   = ['Mean latency (ms)', 'P95 latency (ms)', 'FPS (1000/mean_ms)']
    colors   = [PALETTE_BY_MODEL.get(m, 'steelblue') for m in MODEL_NAMES]

    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    for ax, key, title in zip(axes, metrics_keys, titles):

        for m in MODEL_NAMES + [ "sam3" ]:

          model_base_name = m.split(".")[0].split("-")[0]

          tmp_vals = all_results[m]
          labels  = list(tmp_vals.keys())

          vals = [tmp_vals[l]['summary'][key] for l in labels]
          bars = ax.bar(labels, vals, color=PALETTE_BY_MODEL.get(m, 'steelblue'), width=0.5, edgecolor='white', linewidth=1.5)
          ax.bar_label(bars, fmt=f'{model_base_name} | %.1f ', padding=0, fontweight='bold')
          ax.set_title(title, fontsize=12, fontweight='bold')
          ax.grid(axis='y', alpha=0.3)
          ax.spines[['top','right']].set_visible(False)

    fig.suptitle('Inference |  CPU vs GPU vs SAM3 Summary', fontsize=14, fontweight='bold', y=1.02)
    plt.tight_layout()
    plt.savefig('results/cpu_vs_gpu_vs_sam3.png', dpi=150, bbox_inches='tight')
    plt.show()
else:
    print('Only one device available — skipping CPU vs GPU comparison plot.')


## 4.1 The headline number

**SAM3 on GPU = 1490.7 ms mean → 0.7 FPS.**

It is slower than every YOLO11 variant on CPU except yolo11x, and **~33× slower** than yolo11x on GPU.

**Running SAM3 in real-time is not feasible on current hardware.**

---

### Mean latency comparison

| Model | Hardware | Mean (ms) |
|---|---|---|
| yolo11n | CPU | 135.0 |
| yolo11m | CPU | 730.2 |
| yolo11x | CPU | 1842.5 |
| yolo11n | GPU | 17.9 |
| yolo11m | GPU | 23.4 |
| yolo11x | GPU | 45.7 |
| **SAM3** | **GPU** | **1490.7** |

SAM3 sits between yolo11m and yolo11x on CPU — yet it's running on GPU. The GPU acceleration that collapses YOLO11x from 1842 ms to 45 ms barely moves the needle for SAM3, because the bottleneck is the **ViT image encoder doing global self-attention over 1024²**, which is memory-bandwidth bound rather than compute-parallelism bound.

---

### P95 latency — SAM3 has bad tails too

| Model | Hardware | P95 (ms) |
|---|---|---|
| yolo11x | CPU | 2731.1 |
| **SAM3** | **GPU** | **2092.5** |
| yolo11x | GPU | 50.2 |

SAM3's P95 of **2092 ms** is 40% worse than its mean (1490 ms) — a large tail driven by frame complexity (the spike at frame 200 you saw in the instance count graph). GPU YOLO11 models by contrast have P95 ≈ mean, meaning SAM3's latency is both slow and less predictable.

---

### FPS — the most striking view

| | CPU | GPU | gpu (SAM3) |
|---|---|---|---|
| yolo11n | 7.4 | **55.8** | — |
| yolo11m | 1.4 | **42.7** | — |
| yolo11x | 0.5 | **21.9** | — |
| SAM3 | — | — | **0.7** |

SAM3 at 0.7 FPS lands **below yolo11x on CPU (0.5 FPS)** in the same order of magnitude — but crucially SAM3 achieves this while using a GPU, making it far more expensive in both hardware and power cost per frame.

---

### Why GPU doesn't rescue SAM3

YOLO11 benefits enormously from GPU because its bottleneck is **parallel conv/matmul operations** that GPUs are designed to accelerate.

SAM3's bottleneck is a **1008×1008 global attention ViT encoder** — attention scales quadratically with sequence length, and the memory access pattern is less GPU-friendly. The speedup ratio tells the story:

| Model | CPU→GPU speedup |
|---|---|
| yolo11n | ~7.5× |
| yolo11m | ~31× |
| yolo11x | ~40× |
| SAM3 | not measured (GPU-only here), but architecture-limited |



In [ ]:
from PIL import Image
import matplotlib.pyplot as plt

yolo_11_inference = "results/yolo11_inference_time.png"
sam3_inference = "results/sam3_inference_time.png"

# Open the images
image1 = Image.open(yolo_11_inference)
image2 = Image.open(sam3_inference)

# Display image1 with adapted figsize
plt.figure(figsize=(image1.width / 100, image1.height / 100)) # Convert pixels to inches
plt.imshow(image1)
plt.title('YOLO11 Inference Time')
plt.axis('off') # Turn off axis labels and ticks
plt.show()

# Display image2 with adapted figsize
plt.figure(figsize=(image2.width / 100, image2.height / 100)) # Convert pixels to inches
plt.imshow(image2)
plt.title('SAM3 Inference Time')
plt.axis('off') # Turn off axis labels and ticks
plt.show()

In [ ]:
yolo_11_mask_track = "results/yolo11_masks_tracks.png"
sam3_mask_track = "results/sam3_masks_tracks.png"

# Open the images
image1 = Image.open(yolo_11_mask_track)
image2 = Image.open(sam3_mask_track)

# Display image1 with adapted figsize
plt.figure(figsize=(image1.width / 100, image1.height / 100)) # Convert pixels to inches
plt.imshow(image1)
plt.title('YOLO11 Inference Time')
plt.axis('off') # Turn off axis labels and ticks
plt.show()

# Display image2 with adapted figsize
plt.figure(figsize=(image2.width / 100, image2.height / 100)) # Convert pixels to inches
plt.imshow(image2)
plt.title('SAM3 Inference Time')
plt.axis('off') # Turn off axis labels and ticks
plt.show()

## 4.2 Detected Masks & Active Tracks

---
### Mean = 37.4 masks/frame

Compare this to YOLO11 on the same or similar video:

| Model | Mean detections/frame |
|---|---|
| yolo11n | 3.7 |
| yolo11m | 16.0 |
| yolo11x | 13.9 |
| **SAM3** | **37.4** |

SAM3 detects **~2.3× more instances than yolo11m** and **~10× more than yolo11n**.

This is the core behavioral difference — SAM3 in automatic mode segments everything it can find, with no class filter and no learned suppression of background regions.

---

### Scene structure — same video as the previous YOLO plots

The shape here matches what you saw in the YOLO11 detection graphs for this second video:

| Frame range | SAM3 count | YOLO11m count | Notes |
|---|---|---|---|
| 0–15 | 45–62 | 10–15 | Both peak early |
| 15–35 | Declining to ~30 | Declining | Camera move / scene change |
| 35–75 | 23–44 | 8–15 | Mid-video trough |
| 75–125 | ~25–40 | ~10–18 | Stable, low complexity |
| 125–200 | ~35–45 | ~15–25 | Rising toward end |

The wave shapes correlate — confirming that scene complexity is the shared driver across both models, even though the absolute counts are very different.

---

### The early peak (frames 0–15, up to 62)

The spike to **62 masks at ~frame 10** is the highest point in the entire sequence. This aligns with the SAM3 latency graph from earlier showing ~2100 ms around frames 5–25 — high instance count directly caused high latency. This cross-chart correlation is consistent and clean.


---

# 5 - Conclusions

<u>What this means architecturally</u>
---

SAM3 doesn't classify — it just segments coherent regions. So its count includes things YOLO completely ignores: background texture patches, partial occlusions, shadows, reflections.

**The ~37 mean versus YOLO's ~14 doesn't mean SAM3 is "more accurate" — it means they're solving different problems.**

This chart is a perfect illustration of the **accuracy vs. speed vs. task tradeoff**:

- YOLO11 = class-aware, fast, good enough masks via prototypes
- SAM3 = class-agnostic, exhaustive, near-perfect masks via ViT — but pays ~33× in latency

They are not competing models — they serve different deployment contexts.

The interesting pedagogical point is that **more GPU doesn't fix an architectural mismatch**.

#6 - Think & Exercise

1. Effect of input resolution on detection count and latency

Run yolo11m-seg on the same video at `imgsz=320`,`640`,`960`,`1280` on GPU

* Record mean latency, P95 latency, FPS, and mean detected masks/frame for each
* Plot latency vs imgsz — does it scale quadratically? Fit a curve and check
* At which resolution does detection count stabilize? Is there a point of diminishing returns?

```text
Expected finding: latency scales roughly as imgsz², but mask count plateaus.
The plateau frame is where increasing resolution no longer finds new objects.
```

2. Frame rate subsampling vs. full processing

* Process a 30 FPS video at every frame, every 2nd frame, and every 5th frame using yolo11x-seg on CPU
* Measure effective throughput (detections per real second of video)
* Compare whether skipped frames cause missed detections on fast-moving objects

```
This is a classic real-time tradeoff.
A stride of 2–3 often gives near-identical detection coverage for slow scenes at half the compute cost.
```

3. IoU threshold and NMS sensitivity

- Find a frame with dense, overlapping objects (from the high-complexity frames in the benchmark)

- Run yolo11m-seg with `iou=0.3`, `0.5`, `0.7`, `0.9` and visualize mask output for each.

- **At low IoU, does the model merge adjacent objects?**

- **At high IoU, does it split single objects?**

- Record how detection count changes across IoU values for the same frame

```
Lower IoU threshold = more aggressive suppression = fewer detections.

Dense scenes need higher IoU to keep adjacent detections.

This is exactly the regime that caused latency spikes in the benchmark.
```
